# LLMsを用いる感情分析

## 初期設定

In [ ]:
# !pip install -q datasets openai python-dotenv tiktoken

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics import classification_report, f1_score
import re

In [2]:
# .envからAPI Keyを読み込み
load_dotenv()
#api_key = sk-XXX
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found. Please set it in your .env file.")

client = OpenAI(api_key=api_key)

## データセットの導入

In [3]:
# Load Japanese sentiment dataset
from datasets import load_dataset

dataset = load_dataset("llm-book/wrime-sentiment", remove_neutral=False)
train_df = dataset["train"].to_pandas()
label_feature = dataset["train"].features["label"]
label_names = label_feature.names if hasattr(label_feature, "names") else None
print(f"Label: {label_names}")

Label: ['positive', 'negative', 'neutral']


In [4]:
label_mapping = {0: "POSITIVE", 1: "NEGATIVE", 2: "NEUTRAL"}

In [5]:
sample_data = train_df.copy()
sample_data["gold_sentiment"] = sample_data["label"].apply(lambda x: label_mapping[int(x)] if int(x) in label_mapping else str(x))
sample_data.head()

,sentence,label,user_id,datetime,gold_sentiment
0,ぼけっとしてたらこんな時間。チャリあるから食べにでたいのに…,1,1,2012/7/31 23:48,NEGATIVE
1,今日の月も白くて明るい。昨日より雲が少なくてキレイな〜 と立ち止まる帰り道。チャリなし生活も...,0,1,2012/8/2 23:09,POSITIVE
2,早寝するつもりが飲み物がなくなりコンビニへ。ん、今日、風が涼しいな。,2,1,2012/8/5 0:50,NEUTRAL
3,眠い、眠れない。,1,1,2012/8/8 1:36,NEGATIVE
4,ただいま〜 って新体操してるやん!外食する気満々で家に何もないのに!テレビから離れられない…!,2,1,2012/8/9 22:24,NEUTRAL


In [6]:
print("Sentiment labels:", sorted(sample_data["gold_sentiment"].unique()))
sample_data["gold_sentiment"].value_counts()

Sentiment labels: ['NEGATIVE', 'NEUTRAL', 'POSITIVE']


gold_sentiment
NEGATIVE    10653
NEUTRAL      9851
POSITIVE     9496
Name: count, dtype: int64

## LLMsでの感情分析

- 感情分析を行うことを指示するpromptを用意する
    - 何を・どの形式で」出力してほしいかを明確にするため、感情ラベル（例：POSITIVE / NEUTRAL / NEGATIVE）や回答制約（ラベルのみ返す等）を含む
- 分類対象の文章をプロンプトに差し込み、LLMへ渡す最終的な入力を構成する
- OpenAI API を通じてLLMに入力を送信し、返ってきた出力ラベルを取得して、後続の分析に使える形で記録・格納する

In [7]:
def get_predictions(prompt_generator, texts, model):
  """
  Inference with the API for a model, a list of texts and a prompt format
  """
  results = []
  for i,j in texts.items():
    try:
      print(f"\rRequest element {i}", end= "")
      completion = client.chat.completions.create(
        model=model,
        messages=prompt_generator(j)
      )
      results.append(completion)
    except Exception as e:
      print(e)
      results.append(None)
  print("\rPrediction finished")
  return [i.choices[0].message.content for i in results]


### Zero-shot classification

#### Promptの構成

In [8]:
def build_prompt(text):
  system_prompt = (
      "You are a strict sentiment classifier for Japanese text. "
      "Output exactly one label from: NEGATIVE, NEUTRAL, POSITIVE. "
      "No explanation, no punctuation, no extra words."
  )

  user_prompt = (
      f"Text: {text}\n"
      "Label (NEGATIVE/NEUTRAL/POSITIVE):"
  )

  return [{"role":"system",
           "content":system_prompt,
           },
           {"role":"user",
            "content": user_prompt,
           },
  ]


## Check prompt on an example
text_example = "この映画は期待外れで、時間の無駄だった。"

build_prompt(text_example)

[{'role': 'system',
  'content': 'You are a strict sentiment classifier for Japanese text. Output exactly one label from: NEGATIVE, NEUTRAL, POSITIVE. No explanation, no punctuation, no extra words.'},
 {'role': 'user',
  'content': 'Text: この映画は期待外れで、時間の無駄だった。\nLabel (NEGATIVE/NEUTRAL/POSITIVE):'}]

In [9]:
## Zero-shot: Inspect predictions
r = get_predictions(
    prompt_generator=build_prompt, #prompt you want to use
    texts=sample_data["sentence"][0:5], #texts you want to classify (change or remove [0:5])
    model="gpt-4o-mini" #model you want to use
    )

r

Prediction finished


['NEGATIVE', 'POSITIVE', 'NEUTRAL', 'NEGATIVE', 'NEGATIVE']

###  分類結果の検証

異なるモデルの出力結果を、正解と比較することで分類の精度を評価する

In [10]:
df = sample_data[0:10].copy() #select more or less rows, as you wish

print("gpt-4.1")
df["gpt-4.1"] = get_predictions(
    prompt_generator=build_prompt,
    texts=df["sentence"],
    model="gpt-4.1")
df["gpt-4.1"] = [x.strip().upper() for x in df["gpt-4.1"]]

print("GPT-4.1 mini")
df["gpt41_mini"] = get_predictions(
    prompt_generator=build_prompt,
    texts=df["sentence"],
    model="gpt-4.1-mini")
df["gpt41_mini"] = [x.strip().upper() for x in df["gpt41_mini"]]


gpt-4.1
Prediction finished
GPT-4.1 mini
Prediction finished


In [11]:
print("**** GPT-4.1")
print(classification_report(df["gold_sentiment"], df["gpt-4.1"], digits=3))

print("**** GPT-4.1 mini")
print(classification_report(df["gold_sentiment"], df["gpt41_mini"], digits=3))


**** GPT-4.1
              precision    recall  f1-score   support

    NEGATIVE      0.750     1.000     0.857         3
     NEUTRAL      1.000     0.750     0.857         4
    POSITIVE      1.000     1.000     1.000         3

    accuracy                          0.900        10
   macro avg      0.917     0.917     0.905        10
weighted avg      0.925     0.900     0.900        10

**** GPT-4.1 mini
              precision    recall  f1-score   support

    NEGATIVE      0.600     1.000     0.750         3
     NEUTRAL      1.000     0.500     0.667         4
    POSITIVE      1.000     1.000     1.000         3

    accuracy                          0.800        10
   macro avg      0.867     0.833     0.806        10
weighted avg      0.880     0.800     0.792        10



## データ全体に適用


In [13]:
#full_prediction = get_predictions(
#    prompt_generator=build_prompt_fewshot,
#    texts=headline["headline"],
#    model="gpt-4o-mini")


### Few-shot classification

#### Promptの構成

In [12]:
# Build few-shot examples from the last 5 rows of sample_data
few_shot_examples = list(
    sample_data.tail(5)[["sentence", "gold_sentiment"]].itertuples(index=False, name=None)
 )

def build_prompt_fewshot(text: str, examples: list[tuple] = few_shot_examples):
  system_prompt = (
    "You are a strict Japanese sentiment classifier. "
    "You must respond with one word only — NEGATIVE, NEUTRAL, or POSITIVE. "
    "Do not explain. Do not output anything else."
  )

  examples = "\n".join([f"Classify this text:\n{headline}\nLabel: {label}\n\n" for headline, label in examples])

  user_prompt = f"{examples}\nClassify this text:\n\"{text}\"\nLabel:"

  return [{"role":"system", "content":system_prompt},
          {"role":"user","content": user_prompt}
  ]


build_prompt_fewshot("これはテストです。")

[{'role': 'system',
  'content': 'You are a strict Japanese sentiment classifier. You must respond with one word only — NEGATIVE, NEUTRAL, or POSITIVE. Do not explain. Do not output anything else.'},
 {'role': 'user',
  'content': 'Classify this text:\nそういやギリギリで申し込んだGoToの返金、まだだな\nLabel: NEGATIVE\n\n\nClassify this text:\n私自身はできる限り平日に旅行するんだけど、JRのお得なきっぷや観光列車が土日祝にしか動いてないとかの都合で泣く泣くってことはある\nLabel: NEUTRAL\n\n\nClassify this text:\nちょっと調べてみたら、まだ返金ないっぽいマジか\nLabel: NEUTRAL\n\n\nClassify this text:\nTJO氏が画像で貼ってくれてなかったら私も見られんかったわ。\nLabel: NEUTRAL\n\n\nClassify this text:\n別に個人垢があるから見ようと思えば見えるわけですがそこまでする必要もなかった\nLabel: NEUTRAL\n\n\nClassify this text:\n"これはテストです。"\nLabel:'}]

###  分類結果の検証

Few-shot classificationでは、例を追加することにより分類精度が向上する場合がある。

In [13]:
## Few-shot: validate predictions, to choose model and prompts

### First let's create a smaller dataset
df = sample_data[0:10].copy() #select more or less rows, as you wish

### Then let's predict classes on this smaller dataset
print("GPT-4.1 mini with few-shot prompting")
df["prediction"] = get_predictions(
    prompt_generator=build_prompt_fewshot,
    texts=df["sentence"],
    model="gpt-4.1-mini")
df["prediction"] = [x.strip().upper() for x in df["prediction"]]

### Now let's compute quality scores, comparing with gold standard
print(classification_report(df["gold_sentiment"], df["prediction"], digits=3))


GPT-4.1 mini with few-shot prompting
Prediction finished
              precision    recall  f1-score   support

    NEGATIVE      1.000     0.667     0.800         3
     NEUTRAL      0.800     1.000     0.889         4
    POSITIVE      1.000     1.000     1.000         3

    accuracy                          0.900        10
   macro avg      0.933     0.889     0.896        10
weighted avg      0.920     0.900     0.896        10



## おまけ：費用の概算

APIを利用する場合、基本的には入力トークン数と出力トークン数に応じて課金される。
あくまで概算だが、本例ではプロンプトの入力を800トークン、出力を3トークンとしてカウントする。$N=100{,}000$件のテキストを分類する場合、
[OpenAI GPT API Pricing Calculator](https://gptforwork.com/tools/openai-chatgpt-api-pricing-calculator)で計算さによる試算結果は以下のとおり。
- gpt-5-nano: $3.71
- gpt-5-mini: $18.14
- gpt-5.1: $90.70


## ローカルLLMを用いる感情分類

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [15]:
# MODEL_ID = "Qwen/Qwen2.5-32B-Instruct"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
# MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
      MODEL_ID,
      trust_remote_code=True,
  )

model = AutoModelForCausalLM.from_pretrained(
      MODEL_ID,
      torch_dtype="auto",
      device_map="auto",
      trust_remote_code=True,
      low_cpu_mem_usage=True,
  )

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [16]:
def extract_label(text: str) -> str:
  text = (text or "").strip().upper()
  if text in {"NEGATIVE", "NEUTRAL", "POSITIVE"}:
    return text
  for label in ["NEGATIVE", "NEUTRAL", "POSITIVE"]:
    if re.search(rf"(^|[^A-Z]){label}([^A-Z]|$)", text):
      return label
  return "UNKNOWN"

def get_predictions(prompt_generator, texts, model, tokenizer, max_new_tokens=8):
  results = []
  for i, j in texts.items():
    try:
      print(f"\rRequest element {i}", end="")
      messages = prompt_generator(j)
      chat_text = tokenizer.apply_chat_template(
          messages,
          tokenize=False,
          add_generation_prompt=True,
      )
      model_inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

      output_ids = model.generate(
          **model_inputs,
          max_new_tokens=max_new_tokens,
          do_sample=False,
          pad_token_id=tokenizer.eos_token_id,
      )
      prompt_len = model_inputs["input_ids"].shape[-1]
      generated_ids = output_ids[0][prompt_len:]
      generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
      results.append(extract_label(generated_text))
    except Exception:
      results.append("UNKNOWN")
  print("\rPrediction finished")
  return results

In [17]:
r = get_predictions(
    prompt_generator=build_prompt_fewshot,
    texts=sample_data["sentence"][0:5],
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=16
    )
r

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Prediction finished


['NEGATIVE', 'POSITIVE', 'POSITIVE', 'NEGATIVE', 'POSITIVE']

In [18]:
## Local LLM (few-shot): validate predictions
df_local = sample_data[0:10].copy()  # change size as needed

print("Qwen2.5-1.5B-Instruct")
df_local["local_qwen"] = get_predictions(
    prompt_generator=build_prompt_fewshot,
    texts=df_local["sentence"],
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=16
)

print(classification_report(
    df_local["gold_sentiment"],
    df_local["local_qwen"],
    labels=["NEGATIVE", "NEUTRAL", "POSITIVE"],
    digits=3,
    zero_division=0
))

Qwen2.5-1.5B-Instruct
Prediction finished
              precision    recall  f1-score   support

    NEGATIVE      0.600     1.000     0.750         3
     NEUTRAL      0.000     0.000     0.000         4
    POSITIVE      0.400     0.667     0.500         3

    accuracy                          0.500        10
   macro avg      0.333     0.556     0.417        10
weighted avg      0.300     0.500     0.375        10



| Model                      | Accuracy | Macro-F1 |
| -------------------------- | -------: | -------: |
| Qwen/Qwen2.5-1.5B-Instruct |     0.50 |   0.4167 |
| Qwen/Qwen2.5-7B-Instruct   |     0.70 |   0.6690 |
| Qwen/Qwen2.5-32B-Instruct  |     0.80 |   0.8056 |
